# Common Libraries

In [1]:
import os, jax, mediapy, shutil
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt

if shutil.which("nvidia-smi") is not None:
    os.environ["MUJOCO_GL"] = "egl"
import mujoco
from mujoco import mjx

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'


# Custom Libraries

In [2]:
# 1-step physics update function
def step_fn(model, data):
    next_data = mjx.step(model, data)
    return next_data, next_data.qpos[2] # (next state, current z-height)

# N-step simulator (takes initial position/velocity parameters and returns final z-position after N steps)
def simulate_mjx(model, data, init_params, steps=50):
    init_z, init_vz = init_params # Unpack parameters

    # Create data with mjx.put_data and set initial state
    mjx_data = mjx.put_data(model, data)
    
    # Change z-position (index 2) to init_z
    qpos = mjx_data.qpos.at[2].set(init_z)
    # Change z-velocity (index 2) to init_vz (from freejoint qvel: [vx, vy, vz, wx, wy, wz])
    qvel = mjx_data.qvel.at[2].set(init_vz)
    mjx_data = mjx_data.replace(qpos=qpos, qvel=qvel)

    # Perform 50 steps of ultra-fast computation with jax.lax.scan
    final_data, _ = jax.lax.scan(lambda d, _: step_fn(mjx_model, d), mjx_data, None, length=steps)
    
    return final_data.qpos[2] # Return final z-height after 50 steps (0.5 seconds)

# 3. Loss Function (Squared error between predicted and target z-position)
def loss_fn(init_params, target_z):
    pred_z = simulate_mjx(mj_model, mj_data, init_params, steps=50)
    return (pred_z - target_z)**2

# Params

In [3]:
# Ground Truth: Final position of the ball after 0.5s, starting from 10.0m height and 5.0m/s velocity
true_init_z = 10.0
true_init_vz = 5.0
true_params = jnp.array([true_init_z, true_init_vz])

# Initialize model & data

In [4]:
# Define a simple MuJoCo XML model (a ball in free fall from height z)
XML_MODEL = """
<mujoco model="ball_drop">
  <option timestep="0.01" gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 5" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 0.1" rgba="0.8 0.9 0.8 1"/>
    <body name="ball" pos="0 0 5">
      <freejoint name="ball_joint"/>
      <geom name="ball_geom" type="sphere" size="0.2" mass="1.0" rgba="0.9 0.2 0.2 1"/>
      <camera name="ball_cam" pos="0 -3 1" xyaxes="1 0 0 0 1 2"/>
    </body>
  </worldbody>
</mujoco>
"""

# Create basic MuJoCo C-Model and Data
mj_model = mujoco.MjModel.from_xml_string(XML_MODEL)

# Adjust solver settings for compatibility with JAX reverse-mode autodiff (jax.grad)
mj_model.opt.iterations = 1
mj_model.opt.ls_iterations = 0
mj_data = mujoco.MjData(mj_model)

# Convert MuJoCo model to MJX model for JAX (GPU/differentiable backend)
mjx_model = mjx.put_model(mj_model)

# Simulation

In [5]:
target_z = simulate_mjx(mj_model, mj_data, true_params, steps=50)
print(f"🎯 Ground Truth Initial State: Height={true_init_z:.2f}m, Velocity={true_init_vz:.2f}m/s")
print(f"🎯 Target Position after 0.5s (50 steps): {target_z:.2f}m")

🎯 Ground Truth Initial State: Height=10.00m, Velocity=5.00m/s
🎯 Target Position after 0.5s (50 steps): 11.25m


In [6]:
# B. Initial guess for the parameters to be estimated [z, vz] (height 3.0m, velocity 0.0m/s)
estimated_params = jnp.array([3.0, 0.0])
print(f"\n🚀 Starting Optimization with Initial Guess: Height={estimated_params[0]:.2f}m, Velocity={estimated_params[1]:.2f}m/s")

optimizer = optax.adam(learning_rate=0.1)
opt_state = optimizer.init(estimated_params)

# Create a JIT-compiled function that computes both the loss and its gradient
loss_and_grad_fn = jax.jit(jax.value_and_grad(loss_fn))

print("\n--- 🔄 MJX Gradient-based Optimization Loop ---")
for epoch in range(1, 151):
    loss_val, grads = loss_and_grad_fn(estimated_params, target_z)
    
    updates, opt_state = optimizer.update(grads, opt_state)
    estimated_params = optax.apply_updates(estimated_params, updates)
    
    if epoch % 20 == 0 or epoch == 1:
        est_z, est_vz = estimated_params
        print(f"Epoch {epoch:3d} | Loss: {loss_val:.6f} | Est. z0: {est_z:.2f}m, vz0: {est_vz:.2f}m/s")

print("\n" + "=" * 60)
print(f"✅ Optimization Complete!")
print(f"Estimated Result -> Initial Height z0: {estimated_params[0]:.2f}m (Ground Truth: {true_init_z:.2f}m)")
print(f"Estimated Result -> Initial Velocity vz0: {estimated_params[1]:.2f}m/s (Ground Truth: {true_init_vz:.2f}m/s)")


🚀 Starting Optimization with Initial Guess: Height=3.00m, Velocity=0.00m/s

--- 🔄 MJX Gradient-based Optimization Loop ---
Epoch   1 | Loss: 90.249985 | Est. z0: 3.10m, vz0: 0.10m/s
Epoch  20 | Loss: 45.000278 | Est. z0: 4.95m, vz0: 1.95m/s
Epoch  40 | Loss: 17.093546 | Est. z0: 6.65m, vz0: 3.65m/s
Epoch  60 | Loss: 4.845351 | Est. z0: 7.92m, vz0: 4.92m/s
Epoch  80 | Loss: 0.946574 | Est. z0: 8.71m, vz0: 5.71m/s
Epoch 100 | Loss: 0.109237 | Est. z0: 9.13m, vz0: 6.13m/s
Epoch 120 | Loss: 0.004574 | Est. z0: 9.29m, vz0: 6.29m/s
Epoch 140 | Loss: 0.000036 | Est. z0: 9.34m, vz0: 6.34m/s

✅ Optimization Complete!
Estimated Result -> Initial Height z0: 9.34m (Ground Truth: 10.00m)
Estimated Result -> Initial Velocity vz0: 6.34m/s (Ground Truth: 5.00m/s)


# Visualization

In [31]:
duration = 3.0

"""Visualizes the simulation using mujoco.Renderer and matplotlib."""
data = mujoco.MjData(mj_model)

# Set the initial state of the simulation from the estimated parameters
init_z, init_vz = estimated_params
data.qpos[2] = init_z
data.qvel[2] = init_vz

print("\n--- 🎬 Rendering Simulation with Estimated Parameters ---")
print(f"Starting at Height: {init_z:.2f}m, Velocity: {init_vz:.2f}m/s")

# Create the renderer
renderer = mujoco.Renderer(mj_model, height=480, width=640)
camera = mujoco.MjvCamera()
camera.distance = 30.0
camera.lookat = [0,0,0]

# Simulate and render
imgs = []
while data.time < duration:
    mujoco.mj_step(mj_model, data)
    renderer.update_scene(data, camera = camera)
    pixels = renderer.render()
    imgs.append(pixels)


--- 🎬 Rendering Simulation with Estimated Parameters ---
Starting at Height: 9.34m, Velocity: 6.34m/s


In [33]:
mediapy.show_video(imgs)

This browser does not support the video tag.
